In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")

for f in sorted(RAW.glob("*.csv")):
    size_mb = f.stat().st_size / 1_000_000
    print(f"{f.name:<45} {size_mb:6.1f} MB")

olist_customers_dataset.csv                      9.0 MB
olist_geolocation_dataset.csv                   61.3 MB
olist_order_items_dataset.csv                   15.4 MB
olist_order_payments_dataset.csv                 5.8 MB
olist_order_reviews_dataset.csv                 14.5 MB
olist_orders_dataset.csv                        17.7 MB
olist_products_dataset.csv                       2.4 MB
olist_sellers_dataset.csv                        0.2 MB
product_category_name_translation.csv            0.0 MB


In [2]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")

orders       = pd.read_csv(RAW / "olist_orders_dataset.csv")
order_items  = pd.read_csv(RAW / "olist_order_items_dataset.csv")
payments     = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
reviews      = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
customers    = pd.read_csv(RAW / "olist_customers_dataset.csv")
sellers      = pd.read_csv(RAW / "olist_sellers_dataset.csv")
products     = pd.read_csv(RAW / "olist_products_dataset.csv")
translation  = pd.read_csv(RAW / "product_category_name_translation.csv")
geolocation  = pd.read_csv(RAW / "olist_geolocation_dataset.csv")

tables = {
    "orders": orders, "order_items": order_items, "payments": payments,
    "reviews": reviews, "customers": customers, "sellers": sellers,
    "products": products, "translation": translation, "geolocation": geolocation,
}
print("Loaded", len(tables), "tables")

Loaded 9 tables


In [3]:
summary = pd.DataFrame({
    "rows":    {name: df.shape[0] for name, df in tables.items()},
    "columns": {name: df.shape[1] for name, df in tables.items()},
})
summary

,rows,columns
orders,99441,8
order_items,112650,7
payments,103886,5
reviews,99224,7
customers,99441,5
sellers,3095,4
products,32951,9
translation,71,2
geolocation,1000163,5


In [4]:
for name, df in tables.items():
    print(f"\n=== {name} ===")
    print(df.dtypes.to_string())


=== orders ===
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str

=== order_items ===
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64

=== payments ===
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64

=== reviews ===
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str

=== customers ===
custo

In [5]:
def null_report(df):
    out = pd.DataFrame({
        "nulls": df.isna().sum(),
        "null_pct": (df.isna().mean() * 100).round(2),
    })
    return out[out["nulls"] > 0]

for name, df in tables.items():
    print(f"\n=== {name} ===")
    report = null_report(df)
    print(report.to_string() if len(report) else "no nulls")


=== orders ===
                               nulls  null_pct
order_approved_at                160      0.16
order_delivered_carrier_date    1783      1.79
order_delivered_customer_date   2965      2.98

=== order_items ===
no nulls

=== payments ===
no nulls

=== reviews ===
                        nulls  null_pct
review_comment_title    87656     88.34
review_comment_message  58247     58.70

=== customers ===
no nulls

=== sellers ===
no nulls

=== products ===
                            nulls  null_pct
product_category_name         610      1.85
product_name_lenght           610      1.85
product_description_lenght    610      1.85
product_photos_qty            610      1.85
product_weight_g                2      0.01
product_length_cm               2      0.01
product_height_cm               2      0.01
product_width_cm                2      0.01

=== translation ===
no nulls

=== geolocation ===
no nulls


In [6]:
pk_checks = {
    "orders":      ["order_id"],
    "customers":   ["customer_id"],
    "sellers":     ["seller_id"],
    "products":    ["product_id"],
    "translation": ["product_category_name"],
    "order_items": ["order_id", "order_item_id"],
    "payments":    ["order_id", "payment_sequential"],
    "reviews":     ["review_id"],
}

rows = []
for name, cols in pk_checks.items():
    df = tables[name]
    dup_rows = df.duplicated(subset=cols).sum()
    rows.append({"table": name, "key": cols, "rows": len(df), "duplicate_key_rows": dup_rows})

pd.DataFrame(rows)

,table,key,rows,duplicate_key_rows
0,orders,[order_id],99441,0
1,customers,[customer_id],99441,0
2,sellers,[seller_id],3095,0
3,products,[product_id],32951,0
4,translation,[product_category_name],71,0
5,order_items,"[order_id, order_item_id]",112650,0
6,payments,"[order_id, payment_sequential]",103886,0
7,reviews,[review_id],99224,814


In [7]:
date_cols = {
    "orders": ["order_purchase_timestamp", "order_approved_at",
               "order_delivered_carrier_date", "order_delivered_customer_date",
               "order_estimated_delivery_date"],
    "order_items": ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}

for name, cols in date_cols.items():
    for c in cols:
        s = pd.to_datetime(tables[name][c], errors="coerce")
        print(f"{name}.{c:<35} min={s.min()}   max={s.max()}")

orders.order_purchase_timestamp            min=2016-09-04 21:15:19   max=2018-10-17 17:30:18
orders.order_approved_at                   min=2016-09-15 12:16:38   max=2018-09-03 17:40:06
orders.order_delivered_carrier_date        min=2016-10-08 10:34:01   max=2018-09-11 19:48:28
orders.order_delivered_customer_date       min=2016-10-11 13:46:32   max=2018-10-17 13:22:46
orders.order_estimated_delivery_date       min=2016-09-30 00:00:00   max=2018-11-12 00:00:00
order_items.shipping_limit_date                 min=2016-09-19 00:15:34   max=2020-04-09 22:35:08
reviews.review_creation_date                min=2016-10-02 00:00:00   max=2018-08-31 00:00:00
reviews.review_answer_timestamp             min=2016-10-07 18:32:28   max=2018-10-29 12:27:35


In [8]:
print(orders["order_status"].value_counts(dropna=False), "\n")
print(payments["payment_type"].value_counts(dropna=False), "\n")
print(reviews["review_score"].value_counts(dropna=False).sort_index(), "\n")

print("distinct customer states:", customers["customer_state"].nunique())
print("distinct seller states:  ", sellers["seller_state"].nunique())
print("distinct product categories:", products["product_category_name"].nunique())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64 

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64 

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64 

distinct customer states: 27
distinct seller states:   23
distinct product categories: 73


In [9]:
checks = {
    "order_items.price <= 0":          (order_items["price"] <= 0).sum(),
    "order_items.freight_value < 0":   (order_items["freight_value"] < 0).sum(),
    "payments.payment_value <= 0":     (payments["payment_value"] <= 0).sum(),
    "payments.payment_installments=0": (payments["payment_installments"] == 0).sum(),
    "products.weight_g <= 0":          (products["product_weight_g"] <= 0).sum(),
    "products.length/height/width <= 0": (
        (products[["product_length_cm", "product_height_cm", "product_width_cm"]] <= 0).any(axis=1).sum()
    ),
    "reviews.score not in 1..5":       (~reviews["review_score"].between(1, 5)).sum(),
}
pd.Series(checks, name="rows_flagged")

order_items.price <= 0               0
order_items.freight_value < 0        0
payments.payment_value <= 0          9
payments.payment_installments=0      2
products.weight_g <= 0               4
products.length/height/width <= 0    0
reviews.score not in 1..5            0
Name: rows_flagged, dtype: int64

In [10]:
print("distinct review_ids:", reviews["review_id"].nunique())
print("duplicates on (review_id, order_id):", reviews.duplicated(["review_id", "order_id"]).sum())
print("orders with more than one review:", (reviews.groupby("order_id").size() > 1).sum())

dups = reviews[reviews.duplicated("review_id", keep=False)].sort_values("review_id")
dups.head(6)

distinct review_ids: 98410
duplicates on (review_id, order_id): 0
orders with more than one review: 547


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53


In [12]:
print("distinct customer_id:       ", customers["customer_id"].nunique())
print("distinct customer_unique_id:", customers["customer_unique_id"].nunique())

distinct customer_id:        99441
distinct customer_unique_id: 96096


In [13]:
def fk_coverage(child, child_col, parent, parent_col, label):
    vals = child[child_col].dropna()
    missing = (~vals.isin(parent[parent_col])).sum()
    print(f"{label:<48} missing={missing:>6} of {len(vals):>7} ({missing/len(vals)*100:.2f}%)")

fk_coverage(orders,      "customer_id",          customers,   "customer_id",          "orders.customer_id -> customers")
fk_coverage(order_items, "order_id",             orders,      "order_id",             "order_items.order_id -> orders")
fk_coverage(order_items, "product_id",           products,    "product_id",           "order_items.product_id -> products")
fk_coverage(order_items, "seller_id",            sellers,     "seller_id",            "order_items.seller_id -> sellers")
fk_coverage(payments,    "order_id",             orders,      "order_id",             "payments.order_id -> orders")
fk_coverage(reviews,     "order_id",             orders,      "order_id",             "reviews.order_id -> orders")
fk_coverage(products,    "product_category_name", translation, "product_category_name", "products.category -> translation")
fk_coverage(customers,   "customer_zip_code_prefix", geolocation, "geolocation_zip_code_prefix", "customers.zip -> geolocation")
fk_coverage(sellers,     "seller_zip_code_prefix",   geolocation, "geolocation_zip_code_prefix", "sellers.zip -> geolocation")

orders.customer_id -> customers                  missing=     0 of   99441 (0.00%)
order_items.order_id -> orders                   missing=     0 of  112650 (0.00%)
order_items.product_id -> products               missing=     0 of  112650 (0.00%)
order_items.seller_id -> sellers                 missing=     0 of  112650 (0.00%)
payments.order_id -> orders                      missing=     0 of  103886 (0.00%)
reviews.order_id -> orders                       missing=     0 of   99224 (0.00%)
products.category -> translation                 missing=    13 of   32341 (0.04%)
customers.zip -> geolocation                     missing=   278 of   99441 (0.28%)
sellers.zip -> geolocation                       missing=     7 of    3095 (0.23%)


In [14]:
print("orders without items:   ", (~orders["order_id"].isin(order_items["order_id"])).sum())
print("orders without payments:", (~orders["order_id"].isin(payments["order_id"])).sum())
print("orders without reviews: ", (~orders["order_id"].isin(reviews["order_id"])).sum())

orders without items:    775
orders without payments: 1
orders without reviews:  768


In [15]:
print(order_items.groupby("order_id").size().describe(), "\n")
print(payments.groupby("order_id").size().describe())

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64 

count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64


In [16]:
orders.groupby("order_status")["order_delivered_customer_date"].apply(lambda s: s.isna().sum())

order_status
approved          2
canceled        619
created           5
delivered         8
invoiced        314
processing      301
shipped        1107
unavailable     609
Name: order_delivered_customer_date, dtype: int64

In [17]:
print("distinct zip prefixes:", geolocation["geolocation_zip_code_prefix"].nunique())
print("fully duplicated rows:", geolocation.duplicated().sum())
print("rows per zip prefix:\n", geolocation.groupby("geolocation_zip_code_prefix").size().describe())

distinct zip prefixes: 19015
fully duplicated rows: 261831
rows per zip prefix:
 count    19015.000000
mean        52.598633
std         72.057907
min          1.000000
25%         10.000000
50%         29.000000
75%         66.500000
max       1146.000000
dtype: float64


In [18]:
print("distinct review_ids:", reviews["review_id"].nunique())
print("duplicates on (review_id, order_id):", reviews.duplicated(["review_id", "order_id"]).sum())
print("orders with more than one review:", (reviews.groupby("order_id").size() > 1).sum())

distinct review_ids: 98410
duplicates on (review_id, order_id): 0
orders with more than one review: 547


In [19]:
no_items = orders[~orders["order_id"].isin(order_items["order_id"])]
no_items["order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [20]:
print(orders[~orders["order_id"].isin(payments["order_id"])][["order_id", "order_status"]])

missing_cat = products.loc[
    products["product_category_name"].notna()
    & ~products["product_category_name"].isin(translation["product_category_name"]),
    "product_category_name",
].value_counts()
print(missing_cat)

                               order_id order_status
30710  bfbd0f9bdef84302105ad712db648a6c    delivered
product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64


In [21]:
no_delivery_date = orders[
    (orders["order_status"] == "delivered")
    & orders["order_delivered_customer_date"].isna()
]
print(len(no_delivery_date), "delivered orders with no customer delivery date")
no_delivery_date[["order_id", "order_purchase_timestamp", "order_delivered_carrier_date",
                  "order_estimated_delivery_date"]]

8 delivered orders with no customer delivery date


,order_id,order_purchase_timestamp,order_delivered_carrier_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,2017-11-28 17:44:07,2017-11-30 18:12:23,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,2018-06-20 06:58:43,2018-06-25 08:05:00,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,2018-07-01 17:05:11,2018-07-03 13:57:00,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,2018-07-01 22:05:55,2018-07-03 13:57:00,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,2018-07-01 21:14:02,2018-07-03 09:28:00,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,2017-05-25 23:22:43,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,2018-06-08 12:09:39,2018-06-12 14:10:00,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,2018-06-27 16:09:12,2018-07-03 19:26:00,2018-07-19 00:00:00
